## Gold Layer

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# =========================================================
# Read Silver incrementally
# =========================================================

df_silver = (
    spark.readStream
    .table("fintech.silver.stock_prices")
)

# =========================================================
# Read Gold dimensions
# =========================================================

df_dim_stock = (
    spark.table("fintech.gold.dim_stock")
)

df_dim_date = (
    spark.table("fintech.gold.dim_date")
)


# =========================================================
# Merge Silver batch into Gold fact
# =========================================================

def merge_to_gold(batch_df, batch_id):

    # =====================================================
    # Build fact dataset
    # =====================================================

    df_fact_stock_prices = (
        batch_df.alias("p")

        .join(
            df_dim_stock.alias("s"),
            F.col("p.symbol") == F.col("s.symbol"),
            how="inner"
        )

        .join(
            df_dim_date.alias("d"),
            F.col("p.trade_date") == F.col("d.full_date"),
            how="inner"
        )

        .select(
            F.col("s.stock_key"),
            F.col("d.date_key"),

            F.col("p.open"),
            F.col("p.high"),
            F.col("p.low"),
            F.col("p.close"),
            F.col("p.volume"),
            F.col("p.change"),
            F.col("p.change_percent"),
            F.col("p.vwap"),

            F.col("p._ingestion_timestamp"),
            F.col("p._source_file")
        )
    )

    # =====================================================
    # Merge into Gold fact
    # =====================================================

    fact_table = DeltaTable.forName(
        spark,
        "fintech.gold.fact_stock_prices"
    )

    (
        fact_table.alias("target")
        .merge(
            df_fact_stock_prices.alias("source"),
            """
            target.stock_key = source.stock_key
            AND target.date_key = source.date_key
            """
        )
        .whenMatchedUpdate(
            condition="""
                source._ingestion_timestamp
                > target._ingestion_timestamp
            """,
            set={
                "open": "source.open",
                "high": "source.high",
                "low": "source.low",
                "close": "source.close",
                "volume": "source.volume",
                "change": "source.change",
                "change_percent": "source.change_percent",
                "vwap": "source.vwap",
                "_ingestion_timestamp": "source._ingestion_timestamp",
                "_source_file": "source._source_file"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


# =========================================================
# Start streaming
# =========================================================

query = (
    df_silver.writeStream
    .foreachBatch(merge_to_gold)
    .option(
        "checkpointLocation",
        "/Volumes/fintech/gold/checkpoints/fact_stock_prices/"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()